<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/BART_ZeroShot_DRS_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook runs best on a GPU runtime (e.g., T4 GPU).

# Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import math
import os
import pandas
import sys
from transformers import pipeline

In [ ]:
if IN_COLAB:
  dirpath = '/content/digi-inno-road-prod'
  if not os.path.isdir(dirpath):
    # TODO git pull
    !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git
  sys.path.insert(0,dirpath)
else:
  module_path = os.path.abspath(os.path.join('..'))
  if not module_path in sys.path:
      sys.path.insert(0, module_path)

In [ ]:
from innoprod.digital_readiness_score import DRS_LEVELS
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps

# Read in data

In [ ]:
data = get_sheet_dfs()
# Comment out the line below to explore only the initial dataset (i.e., firms what applied for at least one grant)
roadmaps_df = pandas.concat([data['Roadmaps'], data['RoadmapsWithoutGrants']])
roadmaps_df = wrangle_roadmaps(roadmaps_df)

# Transform data

In [ ]:
qual_cols = [
    'Summary review of Edge Digital diagnostic report & current state and key improvement areas',
    'What are the internal barriers to growth? How do you intend to finance future growth? Are there sufficient leadership and management skills in the business to achieve your growth? What opportunities do you have to expand into new markets?',
    'Details of any existing Digital Strategy',
    'Level of current Strategic Digital Skills/knowledge in the business',
    'Level of current Technical Digital Skills/knowledge in the business',
    'Whether the business is already investing/adopting/utilising Industry 4.0 Technologies, with examples',
    'Summary of the identified problems, including Gap Analysis'
]

In [ ]:
qual_df = roadmaps_df[['Client ID', 'Current Digital Readiness Score (refer to PAS:1040)', 'Number of GAFs'] + qual_cols].copy()
qual_df['Context'] = roadmaps_df.apply(lambda row: ' '.join(filter(None, [row[c] for c in qual_cols])), axis=1)
qual_df.drop(qual_cols, axis=1, inplace=True)
# qual_df


In [ ]:
qual_df = qual_df[qual_df['Current Digital Readiness Score (refer to PAS:1040)'].notna()]
# qual_df

# Run pipeline

In [ ]:
model_name = "facebook/bart-large-mnli"
classifier = pipeline("zero-shot-classification", model=model_name)
hypothesis_template = ("This company's digital readiness level is best described as: {}")

In [ ]:
context = qual_df['Context'].to_list()

results = classifier(
      context,
      candidate_labels=DRS_LEVELS,
      hypothesis_template=hypothesis_template
)

In [ ]:
qual_df['Predicted DRS'] = [DRS_LEVELS.index(rd['labels'][0])+1 for rd in results]
qual_df['Probability'] = [rd['scores'][0] for rd in results]
qual_df['Confidence'] = [abs(math.log(rd['scores'][0]) - math.log(rd['scores'][1])) for rd in results]

In [ ]:
qual_df = qual_df.drop(columns='Context')

In [ ]:
qual_df

# Write results

In [ ]:
if IN_COLAB:
  output_dir = os.path.join(dirpath, 'analysis', 'outputs')

qual_df.to_csv(os.path.join(output_dir, 'BART_DRS_results.csv'), index=False)